In [1]:
import os, json, pickle, joblib, warnings
import numpy as np
import pandas as pd
import tensorflow as tf
import shap
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

CLINICAL_PATH = "/kaggle/input/datasets/chandan294/clinical-assets"
ECG_PATH = "/kaggle/input/notebooks/chandan294/08b-final-training"
FUSION_PATH = "/kaggle/input/notebooks/chandan294/09-confidence-adaptive-fusion"

# --- Clinical assets ---
rf_model = joblib.load(os.path.join(CLINICAL_PATH, "05_random_forest.pkl"))
selected_df = pd.read_csv(os.path.join(CLINICAL_PATH, "04_selected_features.csv"))
CLINICAL_FEATURES = selected_df["Feature"].tolist()

# --- ECG asset ---
cnn_model = tf.keras.models.load_model(os.path.join(ECG_PATH, "results", "models", "08b_best_cnn.keras"))

# ==========================================================
# Recreate Fusion Module Architecture (Mirroring Notebook 9)
# ==========================================================

def confidence(probability):
    """
    Returns model confidence from a calibrated probability.
    Range:
        0.50 -> completely uncertain
        1.00 -> fully confident
    """
    probability = np.asarray(probability)
    return np.maximum(probability, 1 - probability)

def adaptive_fusion(clinical_prob, ecg_prob):
    clinical_conf = confidence(clinical_prob)
    ecg_conf = confidence(ecg_prob)

    weight_clinical = clinical_conf / (clinical_conf + ecg_conf)
    weight_ecg = ecg_conf / (clinical_conf + ecg_conf)

    fused_probability = (weight_clinical * clinical_prob) + (weight_ecg * ecg_prob)

    return {
        "clinical_probability": clinical_prob,
        "ecg_probability": ecg_prob,
        "clinical_confidence": clinical_conf,
        "ecg_confidence": ecg_conf,
        "clinical_weight": weight_clinical,
        "ecg_weight": weight_ecg,
        "fused_probability": fused_probability
    }

# If you also need the gamma-weighted version from Section 8:
def adaptive_fusion_gamma(clinical_prob, ecg_prob, gamma=3):
    clinical_conf = confidence(clinical_prob)
    ecg_conf = confidence(ecg_prob)

    clinical_weight = clinical_conf**gamma
    ecg_weight = ecg_conf**gamma

    clinical_weight = clinical_weight / (clinical_weight + ecg_weight)
    ecg_weight = 1 - clinical_weight

    fused_probability = (clinical_weight * clinical_prob) + (ecg_weight * ecg_prob)

    return {
        "clinical_weight": clinical_weight,
        "ecg_weight": ecg_weight,
        "fused_probability": fused_probability
    }

print("Fusion module loaded/recreated:", type(adaptive_fusion))

print("CLINICAL_FEATURES:", CLINICAL_FEATURES)
print("Assets loaded.")

2026-06-30 07:12:08.284829: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782803528.598124      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782803528.681333      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782803529.373918      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782803529.373970      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782803529.373981      16 computation_placer.cc:177] computation placer alr

Fusion module loaded/recreated: <class 'function'>
CLINICAL_FEATURES: ['cp', 'exang', 'ca', 'sex', 'chol_missing', 'slope_missing', 'thal', 'thalach', 'age', 'oldpeak', 'fbs']
Assets loaded.


In [2]:
class ECGQualityEngine:
    """Heuristic signal-quality checks. Runs before any ECG prediction."""

    def __init__(self, flatline_std_threshold=1e-3, clip_margin=0.001):
        self.flatline_std_threshold = flatline_std_threshold
        self.clip_margin = clip_margin

    def assess(self, signal):
        # signal shape: (timesteps, 12)
        flags = []
        lead_scores = []

        for lead_idx in range(signal.shape[1]):
            lead = signal[:, lead_idx]
            std = np.std(lead)
            lead_flatline = std < self.flatline_std_threshold
            lead_min, lead_max = lead.min(), lead.max()
            saturated = np.mean((lead <= lead_min + self.clip_margin) |
                                 (lead >= lead_max - self.clip_margin)) > 0.3

            if lead_flatline:
                flags.append(f"Lead {lead_idx+1}: flatline detected")
            if saturated:
                flags.append(f"Lead {lead_idx+1}: possible clipping/saturation")

            lead_scores.append(0 if lead_flatline else (50 if saturated else 100))

        # crude noise estimate: ratio of high-frequency energy (diff-based) to total energy
        diffs = np.diff(signal, axis=0)
        noise_ratio = np.mean(np.var(diffs, axis=0)) / (np.mean(np.var(signal, axis=0)) + 1e-8)
        if noise_ratio > 2.0:
            flags.append("High-frequency noise detected")

        quality_score = float(np.mean(lead_scores))
        if noise_ratio > 2.0:
            quality_score = max(0, quality_score - 20)

        return {
            "quality_score": round(quality_score, 1),
            "flags": flags,
            "is_acceptable": quality_score >= 60 and len(flags) <= 2
        }

quality_engine = ECGQualityEngine()
print("ECGQualityEngine ready.")

ECGQualityEngine ready.


In [3]:
class ClinicalEngine:
    def __init__(self, model, feature_order):
        self.model = model
        self.feature_order = feature_order

    def validate_input(self, patient_dict):
        missing = [f for f in self.feature_order if f not in patient_dict]
        if missing:
            raise ValueError(f"Missing clinical features: {missing}")
        return pd.DataFrame([patient_dict])[self.feature_order]

    def predict_raw(self, patient_dict):
        X = self.validate_input(patient_dict)
        raw_prob = self.model.predict_proba(X)[0, 1]
        return float(raw_prob)

clinical_engine = ClinicalEngine(rf_model, CLINICAL_FEATURES)
print("ClinicalEngine ready. Expects keys:", CLINICAL_FEATURES)

ClinicalEngine ready. Expects keys: ['cp', 'exang', 'ca', 'sex', 'chol_missing', 'slope_missing', 'thal', 'thalach', 'age', 'oldpeak', 'fbs']


In [4]:
class ECGEngine:
    def __init__(self, model, expected_shape=(1000, 12)):
        self.model = model
        self.expected_shape = expected_shape

    def validate_input(self, signal):
        signal = np.asarray(signal)
        if signal.shape != self.expected_shape:
            raise ValueError(f"Expected ECG shape {self.expected_shape}, got {signal.shape}")
        return signal

    def predict_raw(self, signal):
        signal = self.validate_input(signal)
        raw_prob = self.model.predict(signal[np.newaxis, ...], verbose=0).ravel()[0]
        return float(raw_prob)

ecg_engine = ECGEngine(cnn_model)
print("ECGEngine ready.")

ECGEngine ready.


In [5]:
class CalibrationEngine:
    """Wraps a fitted Platt scaler. For Clinical, the RF's CalibratedClassifierCV
    is already an end-to-end calibrated model, so this wraps ECG's manual Platt model."""

    def __init__(self, platt_model):
        self.platt_model = platt_model

    def calibrate(self, raw_prob):
        return float(self.platt_model.predict_proba([[raw_prob]])[0, 1])

# Reload ECG's Platt calibrator (lightweight refit on cached val set; see Notebook 09/11)
# In production this object would be saved/loaded directly rather than refit.
from sklearn.model_selection import GroupShuffleSplit

X_full = np.load(os.path.join(ECG_PATH, "cache", "X_full.npy"))
y_full = np.load(os.path.join(ECG_PATH, "cache", "y_full.npy"))
patient_ids = np.load(os.path.join(ECG_PATH, "cache", "patient_ids.npy"))

gss = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=42)
train_idx, temp_idx = next(gss.split(X_full, y_full, groups=patient_ids))
X_temp, y_temp = X_full[temp_idx], y_full[temp_idx]
patients_temp = patient_ids[temp_idx]
gss2 = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=42)
val_idx, _ = next(gss2.split(X_temp, y_temp, groups=patients_temp))
X_val, y_val = X_temp[val_idx], y_temp[val_idx]

raw_val_probs = cnn_model.predict(X_val, verbose=0).ravel()
platt_ecg = LogisticRegression()
platt_ecg.fit(raw_val_probs.reshape(-1, 1), y_val)

ecg_calibration_engine = CalibrationEngine(platt_ecg)
print("CalibrationEngine (ECG) ready.")

I0000 00:00:1782803562.775161      57 service.cc:152] XLA service 0x7cc01000a9d0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782803562.775241      57 service.cc:160]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1782803563.536157      57 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


CalibrationEngine (ECG) ready.


In [6]:
class FusionEngine:
    SEVERITY_BANDS = [
        (0.30, "Low"),
        (0.60, "Moderate"),
        (0.85, "High"),
        (1.01, "Critical")
    ]

    def __init__(self, fusion_fn):
        self.fusion_fn = fusion_fn

    def fuse(self, clinical_prob, ecg_prob):
        result = self.fusion_fn(clinical_prob, ecg_prob)
        fused_prob = result["fused_probability"] if isinstance(result, dict) else result
        return float(fused_prob)

    def get_severity_heuristic(self, fused_prob):
        # IMPORTANT: this is a probability-banded HEURISTIC, not a trained
        # severity model. Real multiclass severity prediction is planned
        # for Notebook 13 and will replace this field once available.
        for threshold, label in self.SEVERITY_BANDS:
            if fused_prob < threshold:
                return label
        return "Critical"

fusion_engine = FusionEngine(adaptive_fusion)
print("FusionEngine ready.")

FusionEngine ready.


In [7]:
class ExplainabilityEngine:
    def __init__(self, rf_model, cnn_model, feature_order):
        self.shap_explainer = shap.TreeExplainer(rf_model)
        self.cnn_model = cnn_model
        self.feature_order = feature_order
        self.last_conv_layer = self._get_last_conv_layer(cnn_model)

    def _get_last_conv_layer(self, model):
        for layer in reversed(model.layers):
            if isinstance(layer, tf.keras.layers.Conv1D):
                return layer.name
        return None

    def explain_clinical(self, patient_dict, top_k=5):
        X = pd.DataFrame([patient_dict])[self.feature_order]
        raw_shap = self.shap_explainer.shap_values(X)
        shap_vals = raw_shap[..., 1][0] if not isinstance(raw_shap, list) else raw_shap[1][0]

        contributions = sorted(
            zip(self.feature_order, shap_vals),
            key=lambda x: abs(x[1]), reverse=True
        )[:top_k]
        return [{"feature": f, "shap_value": round(float(v), 4)} for f, v in contributions]

    def explain_ecg(self, signal, top_k_leads=3, steps=20):
        baseline = np.zeros_like(signal)
        input_tensor = tf.convert_to_tensor(signal[np.newaxis, ...], dtype=tf.float32)
        baseline_tensor = tf.convert_to_tensor(baseline[np.newaxis, ...], dtype=tf.float32)

        alphas = tf.linspace(0.0, 1.0, steps)
        interpolated = tf.concat(
            [baseline_tensor + a * (input_tensor - baseline_tensor) for a in alphas], axis=0
        )

        with tf.GradientTape() as tape:
            tape.watch(interpolated)
            preds = self.cnn_model(interpolated)
            loss = preds[:, 0]

        grads = tape.gradient(loss, interpolated)
        avg_grads = tf.reduce_mean(grads, axis=0)
        ig = (input_tensor[0] - baseline_tensor[0]) * avg_grads
        per_lead_attribution = tf.reduce_sum(tf.abs(ig), axis=0).numpy()

        lead_names = ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"]
        ranked = sorted(zip(lead_names, per_lead_attribution), key=lambda x: x[1], reverse=True)[:top_k_leads]
        return [{"lead": l, "attribution": round(float(v), 6)} for l, v in ranked]

explain_engine = ExplainabilityEngine(rf_model, cnn_model, CLINICAL_FEATURES)
print("ExplainabilityEngine ready.")

ExplainabilityEngine ready.


In [8]:
class RecommendationEngine:
    def generate(self, fused_prob, severity, quality_flags):
        recs = []

        if severity == "Low":
            recs.append("No significant cardiac risk indicators detected. Routine monitoring advised.")
        elif severity == "Moderate":
            recs.append("Some risk indicators present. Recommend follow-up with a cardiologist.")
        elif severity == "High":
            recs.append("Multiple risk indicators present. Prompt cardiology consultation recommended.")
        else:
            recs.append("Strong risk indicators detected. Urgent cardiology evaluation recommended.")

        if quality_flags:
            recs.append("Note: ECG signal quality issues detected — results may be less reliable. Consider re-recording.")

        recs.append("This is a screening tool, not a diagnostic substitute. Consult a licensed physician.")
        return recs

recommendation_engine = RecommendationEngine()
print("RecommendationEngine ready.")

RecommendationEngine ready.


In [9]:
class CardioSensePipeline:
    def __init__(self, clinical_engine, ecg_engine, quality_engine,
                 calibration_engine, fusion_engine, explain_engine, recommendation_engine):
        self.clinical_engine = clinical_engine
        self.ecg_engine = ecg_engine
        self.quality_engine = quality_engine
        self.calibration_engine = calibration_engine
        self.fusion_engine = fusion_engine
        self.explain_engine = explain_engine
        self.recommendation_engine = recommendation_engine

    def run(self, patient_dict, ecg_signal):
        quality_report = self.quality_engine.assess(ecg_signal)

        clinical_raw = self.clinical_engine.predict_raw(patient_dict)
        # Clinical's CalibratedClassifierCV already outputs calibrated probability
        clinical_calibrated = clinical_raw

        ecg_raw = self.ecg_engine.predict_raw(ecg_signal)
        ecg_calibrated = self.calibration_engine.calibrate(ecg_raw)

        fused_prob = self.fusion_engine.fuse(clinical_calibrated, ecg_calibrated)
        severity = self.fusion_engine.get_severity_heuristic(fused_prob)

        clinical_contribution = abs(clinical_calibrated - 0.5)
        ecg_contribution = abs(ecg_calibrated - 0.5)
        total = clinical_contribution + ecg_contribution + 1e-8

        top_clinical_features = self.explain_engine.explain_clinical(patient_dict)
        top_ecg_leads = self.explain_engine.explain_ecg(ecg_signal)

        recommendations = self.recommendation_engine.generate(
            fused_prob, severity, quality_report["flags"]
        )

        return {
            "prediction": "Disease" if fused_prob >= 0.5 else "No Disease",
            "fused_probability": round(fused_prob, 4),
            "severity": severity,
            "severity_source": "heuristic_probability_band",  # NOT a trained severity model
            "confidence": round(max(fused_prob, 1 - fused_prob), 4),
            "branch_contribution": {
                "clinical_pct": round(clinical_contribution / total * 100, 1),
                "ecg_pct": round(ecg_contribution / total * 100, 1)
            },
            "branch_probabilities": {
                "clinical": round(clinical_calibrated, 4),
                "ecg": round(ecg_calibrated, 4)
            },
            "top_clinical_features": top_clinical_features,
            "top_ecg_leads": top_ecg_leads,
            "ecg_quality": quality_report,
            "recommendations": recommendations,
            "disclaimer": "CardioSense AI is a screening aid, not a diagnostic tool. Always consult a licensed physician."
        }

pipeline = CardioSensePipeline(
    clinical_engine, ecg_engine, quality_engine,
    ecg_calibration_engine, fusion_engine, explain_engine, recommendation_engine
)
print("CardioSensePipeline assembled.")

CardioSensePipeline assembled.


In [10]:
# Pull one real clinical row and one real ECG signal from existing test sets for a realistic demo
X_test_full = pd.read_csv(os.path.join(CLINICAL_PATH, "X_test_binary.csv"))
demo_patient = X_test_full[CLINICAL_FEATURES].iloc[5].to_dict()

X_test_ecg, y_test_ecg = X_temp[val_idx], y_temp[val_idx]  # reuse val split signals for demo only
demo_signal = X_val[0]

result = pipeline.run(demo_patient, demo_signal)
print(json.dumps(result, indent=2))

{
  "prediction": "No Disease",
  "fused_probability": 0.3077,
  "severity": "Moderate",
  "severity_source": "heuristic_probability_band",
  "confidence": 0.6923,
  "branch_contribution": {
    "clinical_pct": 24.4,
    "ecg_pct": 75.6
  },
  "branch_probabilities": {
    "clinical": 0.635,
    "ecg": 0.0814
  },
  "top_clinical_features": [
    {
      "feature": "cp",
      "shap_value": -0.1232
    },
    {
      "feature": "thalach",
      "shap_value": 0.0878
    },
    {
      "feature": "thal",
      "shap_value": -0.0786
    },
    {
      "feature": "exang",
      "shap_value": 0.0672
    },
    {
      "feature": "age",
      "shap_value": 0.06
    }
  ],
  "top_ecg_leads": [
    {
      "lead": "aVR",
      "attribution": 0.29747
    },
    {
      "lead": "II",
      "attribution": 0.260415
    },
    {
      "lead": "I",
      "attribution": 0.218637
    }
  ],
  "ecg_quality": {
    "quality_score": 100.0,
    "flags": [],
    "is_acceptable": true
  },
  "recommendation

In [11]:
import shutil

os.makedirs("results12/pipeline", exist_ok=True)

with open("results12/pipeline/cardiosense_pipeline.pkl", "wb") as f:
    pickle.dump({
        "clinical_engine": clinical_engine,
        "ecg_calibration_engine": ecg_calibration_engine,
        "fusion_engine": fusion_engine,
        "quality_engine": quality_engine,
        "recommendation_engine": recommendation_engine,
        "clinical_features": CLINICAL_FEATURES
    }, f)

cnn_model.save("results12/pipeline/ecg_model.keras")
rf_model_path = joblib.dump(rf_model, "results12/pipeline/clinical_rf.pkl")

manifest = {
    "components": ["clinical_engine", "ecg_engine (saved separately, keras)", "quality_engine",
                   "calibration_engine", "fusion_engine", "explainability_engine", "recommendation_engine"],
    "clinical_features": CLINICAL_FEATURES,
    "ecg_input_shape": [1000, 12],
    "severity_source": "heuristic_probability_band — NOT yet a trained multiclass model (planned Notebook 13)"
}
with open("results12/pipeline/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

shutil.make_archive("results12", "zip", "results12")
print("Pipeline artifacts saved.")

Pipeline artifacts saved.
